In [2]:
import torch
import torchvision
import torch.nn as nn
import torchvision.transforms as transforms
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import zipfile

In [3]:
transform = transforms.ToTensor()

with zipfile.ZipFile("C:\\Users\\Marien\\Downloads\\archive.zip", "r") as zip_ref:
    zip_ref.extractall("cats")

dataset = torchvision.datasets.ImageFolder(root="cats", transform=transform)

data_loader = DataLoader(dataset, batch_size=64, shuffle=True)

batch_size = 64

In [ ]:
class Resnet(nn.Module):
    def __init__(self, n_in, n_out, n_group ):
        super().__init__()
        self.n_in = n_in
        self.n_out = n_out
        self.n_group = n_group

        self.Resnet = nn.Sequential(
            nn.GroupNorm(self.n_group, self.n_in),
            nn.SiLU(),
            nn.Conv2d(self.n_in, self.n_out, 3, 1, 1) #p -> p
            )

        self.skip = nn.Conv2d(self.n_in, self.n_out, 1, 1, 0)

        def forward(self, x):
            return self.Resnet(x) + self.skip(x)
        


In [ ]:
class Diffusion_model(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential()

In [5]:
T = 1000

def beta_schedule(timesteps):
    beta_start = 0.0001
    beta_end = 0.02
    return torch.linspace(beta_start, beta_end, timesteps)

beta = beta_schedule(T)
alpha = 1 - beta
alpha_hat = torch.cumprod(alpha, dim=0)

In [ ]:
model = Diffusion_model()
optimizer = optim.Adam(model.parameters(), lr = 0.001)
criterion = nn.MSELoss()

In [ ]:
for epoch in range(10):
    for i, (images, _) in enumerate(data_loader):
        x0 = images
        optimizer.zero_grad()
        index = torch.randint(0, T, (batch_size,))
        alpha_tensor = alpha_hat[index] #tenseur des alpha_hat_t de taille batch_size
        noise = torch.randn_like(x0) #tenseur de bruit de taille batch_size
        xt = torch.sqrt(alpha_tensor) * x0 + torch.sqrt(1 - alpha_tensor) * noise #tenseur de taille batch_size
        predicted_noise = model(xt, index) #tenseur de taille batch_size
        Loss = criterion(predicted_noise, noise)
        Loss.backward()
        optimizer.step()

    print(f"Epoch [{epoch + 1}/10], Loss: {Loss.item():.4f}")
    


